# Reading GHCN observations

This notebook introduces how to use the `hydropandas` package to read, process
and visualise data from the Global Historical Climatology Network (GHCN).
GHCN-Daily is a database of daily climate summaries from land surface stations
around the world, maintained by NOAA.

In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt

import hydropandas as hpd

# enabling logging so we can see what happens in the background
hpd.util.get_color_logger("INFO");

## Read GHCN observations within an extent

Use `hpd.read_ghcn` to download all GHCN stations within a bounding box.
The `extent` parameter is `[xmin, xmax, ymin, ymax]` in the coordinate system
given by `crs`. Setting `elements='PRCP'` downloads only precipitation data;
leave it as `None` to retrieve all available elements.

In [ ]:
# read GHCN precipitation observations for an area in the Netherlands
# extent: [xmin, xmax, ymin, ymax] in RD New (EPSG:28992)
extent = [130_000, 150_000, 450_000, 470_000]

oc = hpd.read_ghcn(
    extent=extent,
    crs=28992,
    elements="PRCP",
    tmin="2020-01-01",
    tmax="2021-12-31",
)
oc

In [ ]:
# plot station locations on a map
gdf = oc.to_gdf().to_crs(epsg=4326)
ax = gdf.plot(figsize=(8, 8), color="steelblue", markersize=60)
ctx.add_basemap(ax=ax, crs=4326, attribution=False)

for idx, row in gdf.iterrows():
    ax.annotate(
        text=idx,
        xy=(row.geometry.x, row.geometry.y),
        fontsize=7,
        ha="center",
        va="bottom",
    )
ax.set_title("GHCN stations")
plt.tight_layout()

## Plot observations from a single station

In [ ]:
# select the first observation and plot the precipitation time series
o = oc.iloc[0].obs
print(f"Station: {o.name}  |  element: PRCP  |  unit: {o.unit}")
o["PRCP"].plot(
    figsize=(12, 4),
    drawstyle="steps",
    ylabel="Precipitation (m)",
    title=f"Daily precipitation – {o.name}",
)
plt.tight_layout()

## Download multiple elements (temperature)

Omit the `elements` argument (or pass a list) to download all available
elements for the stations in the extent. Here we request maximum and minimum
daily temperature (`TMAX` and `TMIN`) for a small extent around De Bilt.

In [ ]:
# read TMAX and TMIN for De Bilt area
extent_debilt = [4.9, 5.2, 51.9, 52.1]  # [xmin, xmax, ymin, ymax] WGS84

oc_temp = hpd.read_ghcn(
    extent=extent_debilt,
    crs=4326,
    elements=["TMAX", "TMIN"],
    tmin="2020-01-01",
    tmax="2020-12-31",
)
oc_temp

In [ ]:
# plot temperature observations for the first station
temp_min = oc_temp.get_obs(meteo_var="TMIN", station="NLE00152476")
temp_max = oc_temp.get_obs(meteo_var="TMAX", station="NLE00152476")
f, ax = plt.subplots(figsize=(12, 4))
temp_max["TMAX"].plot(ax=ax, label="TMAX", color="tomato")
temp_min["TMIN"].plot(ax=ax, label="TMIN", color="steelblue")
ax.set_ylabel("Temperature (°C / 0.1 °C)")
ax.set_title(f"Daily temperature – {temp_min.station}")
ax.legend()
f.tight_layout()

## Explore available stations

Use `ghcn.get_stations` to retrieve a GeoDataFrame with all GHCN stations
worldwide or filtered to an extent, without downloading any measurements.

In [ ]:
# get station metadata only (fast – no measurements downloaded)
oc_meta = hpd.read_ghcn(
    extent=extent_debilt,
    crs=4326,
    only_metadata=True,
)
print(f"Found {len(oc_meta)} stations in the extent")
oc_meta[["x", "y", "source", "station"]]

In [ ]:
# interactive map of all downloaded observations
oc.plots.interactive_map(plot_dir="figure", per_location=False, popup_width=300)